In [1]:
import os
os.chdir(r'D:\8th Semester\Machine Learning Lab\data parisr')

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [89]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [90]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [91]:
model1 = MLP()
model1.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_7 (Flatten)         (None, 504)               0         


                                                                 
 dense_14 (Dense)            (None, 32)                16160     
                                                                 
 dense_15 (Dense)            (None, 1)                 33        
                                                                 
Total params: 16193 (63.25 KB)
Trainable params: 16193 (63.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [92]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [93]:
checkpoints = r'D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\8th Semester\Machine Learning Lab\data parisr'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [94]:
os.path.exists(JSON_PATH)

True

In [95]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [96]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [97]:
import os
path_dataset =r'D:\8th Semester\Machine Learning Lab\data parisr'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [98]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.029432296752929688 sec


In [99]:
train_X.shape

(835, 24, 21)

In [100]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/60
26/27 [===========================>..] - ETA: 0s - loss: 0.3724 - mae: 0.3724 - mape: 202.2413
Epoch 1: val_loss improved from inf to 0.12040, saving model to D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0001-loss0.12.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 3s 51ms/step - loss: 0.3716 - mae: 0.3716 - mape: 202.0254 - val_loss: 0.1204 - val_mae: 0.1204 - val_mape: 38.7151
Epoch 2/60
20/27 [=====================>........] - ETA: 0s - loss: 0.1077 - mae: 0.1077 - mape: 53.3931
Epoch 2: val_loss improved from 0.12040 to 0.09466, saving model to D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0002-loss0.09.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 20ms/step - loss: 0.1019 - mae: 0.1019 - mape: 51.8775 - val_loss: 0.0947 - val_mae: 0.0947 - val_mape: 33.3226
Epoch 3/60
18/27 [===================>..........] - ETA: 0s - loss: 0.0817 - mae: 0.0817 - mape: 48.5496
Epoch 3: val_loss improved from 0.09466 to 0.06772, saving model to D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0003-loss0.07.h5


...

27/27 [==============================] - 0s 17ms/step - loss: 0.0468 - mae: 0.0468 - mape: 24.9812 - val_loss: 0.0335 - val_mae: 0.0335 - val_mape: 10.5039
Epoch 26/60
14/27 [==============>...............] - ETA: 0s - loss: 0.0380 - mae: 0.0380 - mape: 19.4553
Epoch 26: val_loss did not improve from 0.03349
27/27 [==============================] - 1s 19ms/step - loss: 0.0445 - mae: 0.0445 - mape: 22.3888 - val_loss: 0.0357 - val_mae: 0.0357 - val_mape: 11.8182
Epoch 27/60
19/27 [====================>.........] - ETA: 0s - loss: 0.0374 - mae: 0.0374 - mape: 18.2672
Epoch 27: val_loss did not improve from 0.03349
27/27 [==============================] - 1s 21ms/step - loss: 0.0349 - mae: 0.0349 - mape: 17.7899 - val_loss: 0.0379 - val_mae: 0.0379 - val_mape: 12.4602
Epoch 28/60
15/27 [===============>..............] - ETA: 0s - loss: 0.0376 - mae: 0.0376 - mape: 22.9899
Epoch 28: val_loss did not improve from 0.03349
27/27 [==============================] - 1s 21ms/step - loss: 0.0375 -

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 39ms/step - loss: 0.0289 - mae: 0.0289 - mape: 15.4057 - val_loss: 0.0290 - val_mae: 0.0290 - val_mape: 8.2931
Epoch 34/60
21/27 [======================>.......] - ETA: 0s - loss: 0.0281 - mae: 0.0281 - mape: 13.9389
Epoch 34: val_loss did not improve from 0.02898
27/27 [==============================] - 1s 24ms/step - loss: 0.0284 - mae: 0.0284 - mape: 13.5168 - val_loss: 0.0825 - val_mae: 0.0825 - val_mape: 28.7107
Epoch 35/60
21/27 [======================>.......] - ETA: 0s - loss: 0.0539 - mae: 0.0539 - mape: 25.4563
Epoch 35: val_loss did not improve from 0.02898
27/27 [==============================] - 1s 47ms/step - loss: 0.0504 - mae: 0.0504 - mape: 24.7486 - val_loss: 0.0403 - val_mae: 0.0403 - val_mape: 12.8105
Epoch 36/60
19/27 [====================>.........] - ETA: 0s - loss: 0.0367 - mae: 0.0367 - mape: 21.5770
Epoch 36: val_loss did not improve from 0.02898
27/27 [==============================] - 1s 26ms/step - loss: 0.0357 - 

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 25ms/step - loss: 0.0329 - mae: 0.0329 - mape: 14.7904 - val_loss: 0.0286 - val_mae: 0.0286 - val_mape: 8.0363


In [101]:

model = load_model(r'D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0001-loss0.13.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 165ms/step
Mean Absolute Error (MAE): 2235.21
Median Absolute Error (MedAE): 2345.64
Mean Squared Error (MSE): 5244888.88
Root Mean Squared Error (RMSE): 2290.17
Mean Absolute Percentage Error (MAPE): 14.29 %
Median Absolute Percentage Error (MDAPE): 14.83 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# Fine Tuning

In [105]:
checkpoints = r'D:\8th Semester\Machine Learning Lab\data parisr-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'D:\8th Semester\Machine Learning Lab\data parisr\\E1-cp-0054-loss0.03.h5'
start_epoch= 56

In [106]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading D:\8th Semester\Machine Learning Lab\data parisr\\E1-cp-0054-loss0.03.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [107]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
15/27 [===============>..............] - ETA: 0s - loss: 0.0233 - mae: 0.0233 - mape: 12.1313 
Epoch 1: val_loss improved from inf to 0.04338, saving model to D:\8th Semester\Machine Learning Lab\data parisr-0001-loss0.04.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 2s 45ms/step - loss: 0.0233 - mae: 0.0233 - mape: 11.8044 - val_loss: 0.0434 - val_mae: 0.0434 - val_mape: 12.1856
Epoch 2/10
22/27 [=======================>......] - ETA: 0s - loss: 0.0193 - mae: 0.0193 - mape: 8.0937
Epoch 2: val_loss improved from 0.04338 to 0.04124, saving model to D:\8th Semester\Machine Learning Lab\data parisr-0002-loss0.04.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 24ms/step - loss: 0.0191 - mae: 0.0191 - mape: 8.3775 - val_loss: 0.0412 - val_mae: 0.0412 - val_mape: 12.0175
Epoch 3/10
24/27 [=========================>....] - ETA: 0s - loss: 0.0193 - mae: 0.0193 - mape: 8.7153
Epoch 3: val_loss improved from 0.04124 to 0.03680, saving model to D:\8th Semester\Machine Learning Lab\data parisr-0003-loss0.04.h5


...

27/27 [==============================] - 1s 35ms/step - loss: 0.0195 - mae: 0.0195 - mape: 8.6592 - val_loss: 0.0368 - val_mae: 0.0368 - val_mape: 10.3911
Epoch 4/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0183 - mae: 0.0183 - mape: 8.0157
Epoch 4: val_loss did not improve from 0.03680
27/27 [==============================] - 0s 19ms/step - loss: 0.0184 - mae: 0.0184 - mape: 8.3326 - val_loss: 0.0369 - val_mae: 0.0369 - val_mape: 10.3296
Epoch 5/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0188 - mae: 0.0188 - mape: 8.6629
Epoch 5: val_loss improved from 0.03680 to 0.03634, saving model to D:\8th Semester\Machine Learning Lab\data parisr-0005-loss0.04.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 20ms/step - loss: 0.0188 - mae: 0.0188 - mape: 8.6333 - val_loss: 0.0363 - val_mae: 0.0363 - val_mape: 10.0875
Epoch 6/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0177 - mae: 0.0177 - mape: 7.9043
Epoch 6: val_loss improved from 0.03634 to 0.03502, saving model to D:\8th Semester\Machine Learning Lab\data parisr-0006-loss0.04.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 24ms/step - loss: 0.0179 - mae: 0.0179 - mape: 7.8956 - val_loss: 0.0350 - val_mae: 0.0350 - val_mape: 9.8858
Epoch 7/10
23/27 [========================>.....] - ETA: 0s - loss: 0.0180 - mae: 0.0180 - mape: 8.3833
Epoch 7: val_loss did not improve from 0.03502
27/27 [==============================] - 1s 55ms/step - loss: 0.0184 - mae: 0.0184 - mape: 8.1689 - val_loss: 0.0422 - val_mae: 0.0422 - val_mape: 12.3532
Epoch 8/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0182 - mae: 0.0182 - mape: 8.1298
Epoch 8: val_loss did not improve from 0.03502
27/27 [==============================] - 1s 22ms/step - loss: 0.0182 - mae: 0.0182 - mape: 8.1221 - val_loss: 0.0392 - val_mae: 0.0392 - val_mape: 11.3613
Epoch 9/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0181 - mae: 0.0181 - mape: 8.2034
Epoch 9: val_loss did not improve from 0.03502
27/27 [==============================] - 1s 27ms/step - loss: 0.0180 - mae: 0.0180 

In [108]:

model = load_model(r'D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0001-loss0.13.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 96ms/step
Mean Absolute Error (MAE): 2235.21
Median Absolute Error (MedAE): 2345.64
Mean Squared Error (MSE): 5244888.88
Root Mean Squared Error (RMSE): 2290.17
Mean Absolute Percentage Error (MAPE): 14.29 %
Median Absolute Percentage Error (MDAPE): 14.83 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
